# Pipeline Run Audit — `b2b_ecommerce.audit.pipeline_audit`

**One row per pipeline update × source-to-target flow** — every hop data makes through the platform:

| Pipeline | Layer | Flows | load_type | scd_type |
| --- | --- | --- | --- | --- |
| `raw_files_to_bronze_etl` | bronze | 2 (raw volume files → leads, logs) | `incremental_append` | — |
| `postgres_to_bronze` | bronze | 7 (Postgres → connect tables) | `incremental_cdc` | — (CDC keys + cursor read live from connector table properties) |
| `bonze_to_silver_etl` | silver | 9 (bronze → silver) | `incremental_merge` | `type_2` ×7, `type_1` (leads, logs) |

Plus the two quarantine tables (`leads_silver`, `logs_silver`), whose writes are counted in
`rows_quarantined` of their silver flow.

**Where each metric comes from:**

* `rows_read` / `rows_skipped` — pipeline event log (`stream_progress` expectation metrics, dataset level)
* `rows_inserted` / `rows_updated` / `rows_deleted` — Delta history of the target table inside the update's time window (`numTargetRowsInserted/Updated/Deleted` for MERGE, `numOutputRows` for appends)
* `rows_quarantined` — Delta history of the quarantine table inside the same window
* `source_max_ts` — max `updated_at` currently in the target (set for a pipeline's latest update; NULL on backfilled history)
* `cdc_keys` — connector primary key + cursor (connect flows, read live) or the AUTO CDC keys + sequence column (silver flows)

A row is only written for flows the update actually **touched** (batch metrics or Delta commits in the window).

**Incremental + self-backfilling**: updates already recorded (`pipeline_run_id` + `target_table_name`) are skipped — the first run backfills the entire history from the three event logs, later runs only add new updates.

**History:** the pre-2026-11-16 table-snapshot version of this audit is preserved as `audit.ingestion_audit_legacy`.


In [0]:
# Pipeline run audit engine - config + run_pipeline_audit().
# Writes b2b_ecommerce.audit.pipeline_audit: ONE ROW PER PIPELINE UPDATE
# x SOURCE-TO-TARGET FLOW (only flows the update actually touched).
# Data sources: pipeline event logs (rows_read/rows_skipped per batch),
# Delta table history (rows inserted/updated/deleted per commit window),
# connector table properties (CDC keys + cursor, appended to the source
# name), target tables (source_max_ts). Incremental: already-recorded
# (pipeline_run_id + target_table_name) pairs are skipped, so the first
# run backfills the full history and later runs only add new updates.
import json, uuid
from datetime import datetime, timezone
from collections import defaultdict
from pyspark.sql import functions as F

AUDIT_TABLE = "b2b_ecommerce.audit.pipeline_audit"

PIPELINES = [
    dict(name="raw_files_to_bronze_etl", pid="1fec3db6-39c5-4482-a15b-634fdcfd4414",
         log="b2b_ecommerce.bronze.event_log_1fec3db6_39c5_4482_a15b_634fdcfd4414",
         layer="bronze",
         flows=[
             dict(source="raw volume files (leads)", target="b2b_ecommerce.bronze.leads",
                  short="leads", load_type="incremental_append", scd_type=None),
             dict(source="raw volume files (logs)", target="b2b_ecommerce.bronze.logs",
                  short="logs", load_type="incremental_append", scd_type=None),
         ]),
    dict(name="postgres_to_bronze", pid="9fea88a8-6a24-43f4-a698-98904f2737f8",
         log="b2b_ecommerce.bronze.event_log_9fea88a8_6a24_43f4_a698_98904f2737f8",
         layer="bronze",
         flows=[dict(source=f"postgres.b2b_ecommerce_bronze.{t}", target=f"b2b_ecommerce.bronze.{t}",
                     short=t, load_type="incremental_cdc", scd_type=None)
                for t in ["companies", "customers", "orders", "order_items",
                          "products", "suppliers", "supplier_products"]]),
    dict(name="bonze_to_silver_etl", pid="9a6f6758-47e4-4000-a468-08c8f534da3d",
         log="b2b_ecommerce.silver.event_log_9a6f6758_47e4_4000_a468_08c8f534da3d",
         layer="silver",
         flows=[
             dict(source="b2b_ecommerce.bronze.companies (merge [company_id] seq=updated_at)",
                  target="b2b_ecommerce.silver.companies", short="companies",
                  load_type="incremental_merge", scd_type="type_2"),
             dict(source="b2b_ecommerce.bronze.customers (merge [customer_id] seq=updated_at)",
                  target="b2b_ecommerce.silver.customers", short="customers",
                  load_type="incremental_merge", scd_type="type_2"),
             dict(source="b2b_ecommerce.bronze.orders (merge [order_id] seq=updated_at)",
                  target="b2b_ecommerce.silver.orders", short="orders",
                  load_type="incremental_merge", scd_type="type_2"),
             dict(source="b2b_ecommerce.bronze.order_items (merge [order_item_id] seq=updated_at)",
                  target="b2b_ecommerce.silver.order_items", short="order_items",
                  load_type="incremental_merge", scd_type="type_2"),
             dict(source="b2b_ecommerce.bronze.products (merge [product_id] seq=updated_at)",
                  target="b2b_ecommerce.silver.products", short="products",
                  load_type="incremental_merge", scd_type="type_2"),
             dict(source="b2b_ecommerce.bronze.suppliers (merge [supplier_id] seq=updated_at)",
                  target="b2b_ecommerce.silver.suppliers", short="suppliers",
                  load_type="incremental_merge", scd_type="type_2"),
             dict(source="b2b_ecommerce.bronze.supplier_products (merge [supplier_id, product_id] seq=updated_at)",
                  target="b2b_ecommerce.silver.supplier_products", short="supplier_products",
                  load_type="incremental_merge", scd_type="type_2"),
             dict(source="b2b_ecommerce.bronze.leads (merge [lead_id] seq=updated_at)",
                  target="b2b_ecommerce.silver.leads", short="leads",
                  load_type="incremental_merge", scd_type="type_1",
                  quar="b2b_ecommerce.quarantine.leads_silver"),
             dict(source="b2b_ecommerce.bronze.logs (merge [log_id = sha2(source_file|log_line)] seq=updated_at)",
                  target="b2b_ecommerce.silver.logs", short="logs",
                  load_type="incremental_merge", scd_type="type_1",
                  quar="b2b_ecommerce.quarantine.logs_silver"),
         ]),
]


def _resolve(name, flows):
    """Map an event-log dataset/stream name to a flow (longest short name first)."""
    for f in sorted(flows, key=lambda f: -len(f["short"])):
        if f["short"] in name:
            return f
    return None


def _update_windows(log):
    """(update_id, first event ts, last event ts) for every update, oldest first."""
    df = spark.sql(
        f"SELECT origin.update_id uid, min(timestamp) s, max(timestamp) e "
        f"FROM {log} WHERE origin.update_id IS NOT NULL GROUP BY origin.update_id")
    return sorted([(r["uid"], r["s"], r["e"]) for r in df.collect()], key=lambda w: w[1])


def _batch_metrics(log, flows):
    """(update_id, target) -> [rows_read, rows_skipped] from stream_progress expectation metrics."""
    out = defaultdict(lambda: [0, 0])
    rows = spark.sql(
        f"SELECT origin.update_id uid, details FROM {log} "
        "WHERE event_type = 'stream_progress' AND details IS NOT NULL").collect()
    for r in rows:
        try:
            pj = json.loads(r["details"])["stream_progress"]["progress_json"]
            m = json.loads(pj)
        except Exception:
            continue
        ds = m.get("name", "")
        if "b2b_ecommerce.quarantine." in ds:
            continue  # quarantine streams are counted via their table history
        f = _resolve(ds, flows)
        if not f:
            continue
        for k, v in (m.get("observedMetrics") or {}).items():
            if isinstance(v, dict) and k.count(".") >= 2:  # dataset-level metric
                out[(r["uid"], f["target"])][0] += v.get("total") or 0
                out[(r["uid"], f["target"])][1] += v.get("dropped") or 0
    return out


def _history(table):
    """Delta history rows: [(ts, operation, {metrics})]."""
    try:
        return [(r["timestamp"], r["operation"], r["operationMetrics"] or {})
                for r in spark.sql(f"DESCRIBE HISTORY {table}")
                .select("timestamp", "operation", "operationMetrics").collect()]
    except Exception:
        return []


def _delta_in_window(hist, s, e):
    """(inserted, updated, deleted) summed over the commits inside [s, e]."""
    ins = upd = dele = 0
    for ts, op, m in hist:
        if s <= ts <= e:
            ins += int(m.get("numTargetRowsInserted") or 0)
            upd += int(m.get("numTargetRowsUpdated") or 0)
            dele += int(m.get("numTargetRowsDeleted") or 0)
            if op != "MERGE":
                ins += int(m.get("numOutputRows") or 0)
    return ins, upd, dele


def _connector_cdc(target):
    """CDC basis of a connect-managed table, read live from its properties."""
    try:
        props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {target}").collect()}
        pk = props.get("__ingestion_connector_primary_key", "[]").replace("[", "").replace("]", "").replace('"', "")
        cur = props.get("__ingestion_connector_cursor_columns", "[]").replace("[", "").replace("]", "").replace('"', "")
        return f" (CDC keys [{pk}], cursor {cur})"
    except Exception:
        return ""


def run_pipeline_audit():
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ-") + uuid.uuid4().hex[:6]
    existing = {(r["pipeline_run_id"], r["target_table_name"]) for r in
                spark.sql(f"SELECT pipeline_run_id, target_table_name FROM {AUDIT_TABLE}").collect()}
    hist_cache = {}
    rows_out = []

    for p in PIPELINES:
        flows = p["flows"]
        for f in flows:
            if f["target"] not in hist_cache:
                hist_cache[f["target"]] = _history(f["target"])
            q = f.get("quar")
            if q and q not in hist_cache:
                hist_cache[q] = _history(q)
            # connect flows: append the CDC basis (keys + cursor) to the source name
            if f["load_type"] == "incremental_cdc":
                f["source"] = f["source"].split(" (CDC")[0] + _connector_cdc(f["target"])

        wins = _update_windows(p["log"])
        if not wins:
            continue
        bm = _batch_metrics(p["log"], flows)

        for uid, s, e in wins:
            for f in flows:
                key = (uid, f["target"])
                if key in existing:
                    continue
                read, skipped = bm.get(key, (None, None))
                ins, upd, dele = _delta_in_window(hist_cache[f["target"]], s, e)
                quar = 0
                q = f.get("quar")
                if q:
                    quar = sum(int(m.get("numOutputRows") or 0)
                               for ts, op, m in hist_cache[q] if s <= ts <= e)
                touched = ((read or 0) or (skipped or 0) or ins or upd or dele or quar)
                if not touched:
                    continue
                rows_out.append([
                    run_id, uid, p["name"], p["layer"], f["source"], f["target"],
                    f["load_type"], f["scd_type"], s, e,
                    read, ins, upd, dele, skipped, quar, None])
                existing.add(key)

    # source_max_ts: the freshest source change in each target is attached
    # to that flow's most recent audited update (a pipeline's latest update
    # may be a no-op with no data movement at all).
    last_idx = {}
    for i, r in enumerate(rows_out):
        last_idx[r[5]] = i
    for tgt, i in last_idx.items():
        try:
            rows_out[i][16] = spark.sql(f"SELECT max(updated_at) m FROM {tgt}").collect()[0]["m"]
        except Exception:
            pass

    if rows_out:
        schema = ("audit_run_id string, pipeline_run_id string, pipeline_name string, layer string, "
                  "source_table_name string, target_table_name string, load_type string, "
                  "scd_type string, started_at timestamp, finished_at timestamp, "
                  "rows_read bigint, rows_inserted bigint, rows_updated bigint, rows_deleted bigint, "
                  "rows_skipped bigint, rows_quarantined bigint, source_max_ts timestamp")
        spark.createDataFrame(rows_out, schema).write.mode("append").saveAsTable(AUDIT_TABLE)

    summary = (spark.table(AUDIT_TABLE).filter(F.col("audit_run_id") == run_id)
               .groupBy("pipeline_name", "layer")
               .agg(F.count("*").alias("flow_rows"), F.min("started_at").alias("earliest_update"),
                    F.max("finished_at").alias("latest_update"))
               .orderBy("pipeline_name"))
    print(f"Pipeline audit run {run_id}: {len(rows_out)} update-flow rows written")
    return run_id, len(rows_out), summary


In [0]:
# Bootstrap + run.
spark.sql("""CREATE TABLE IF NOT EXISTS b2b_ecommerce.audit.pipeline_audit (
  audit_run_id      STRING COMMENT 'Audit execution that wrote this row',
  pipeline_run_id   STRING COMMENT 'Pipeline update id (the actual run)',
  pipeline_name     STRING COMMENT 'raw_files_to_bronze_etl | postgres_to_bronze | bonze_to_silver_etl',
  layer             STRING COMMENT 'bronze | silver',
  source_table_name STRING COMMENT 'Source: raw file set, Postgres table (+ CDC keys/cursor), or bronze table (+ merge keys)',
  target_table_name STRING COMMENT 'Full 3-part target table',
  load_type         STRING COMMENT 'incremental_append | incremental_cdc | incremental_merge',
  scd_type          STRING COMMENT 'type_1 | type_2 | NULL (silver only)',
  started_at        TIMESTAMP COMMENT 'Update start (first event ts)',
  finished_at       TIMESTAMP COMMENT 'Update finish (last event ts)',
  rows_read         BIGINT COMMENT 'Rows the update pushed through this flow',
  rows_inserted     BIGINT COMMENT 'Rows newly inserted into target (Delta history)',
  rows_updated      BIGINT COMMENT 'Rows updated in place (SCD merge)',
  rows_deleted      BIGINT COMMENT 'Rows deleted from target',
  rows_skipped      BIGINT COMMENT 'Rows dropped by drop-level expectations (not in target)',
  rows_quarantined  BIGINT COMMENT 'Rows written to the matching quarantine table in this update',
  source_max_ts     TIMESTAMP COMMENT 'Max source change timestamp (updated_at) seen in target'
) COMMENT 'Pipeline run audit: one row per pipeline update x source-to-target flow, all layers'""")

run_id, n_rows, summary = run_pipeline_audit()
display(summary)
